	1. Load Data: Load the provided dataset into a Pandas DataFrame.
	2. Clean: Briefly describe the data cleaning steps you'd take. Then, perform one or two of the most critical cleaning steps (e.g., handling missing values in a key column, converting a column to the correct data type).
	3. Analyze: Calculate a key metric that would be valuable for a business stakeholder. Define this metric and present your calculation with a clear output. Explain the business value of this metric.


In [34]:
import pandas as pd

# 1️⃣ Load the data

csv_path = r"C:\Users\bhrys\loan.csv"
df = pd.read_csv(csv_path, low_memory=False)

In [9]:
print("✅ Data loaded successfully")
print("Rows:", len(df), "| Columns:", len(df.columns))


✅ Data loaded successfully
Rows: 2260668 | Columns: 145


In [ ]:
# 2️⃣ Parse issue_d ('Dec-2018' format)

df['issue_d_parsed'] = pd.to_datetime(df['issue_d'], format='%b-%Y', errors='coerce')

In [14]:
# 3️⃣ Normalize loan_status
def normalize_status(s):
    if pd.isna(s):
        return 'unknown'
    s = str(s).strip().lower()
    if 'charged off' in s:
        return 'charged_off'
    if 'default' in s and 'charged off' not in s:
        return 'default'
    if 'fully paid' in s:
        return 'fully_paid'
    if 'current' in s:
        return 'current'
    if 'late' in s:
        return 'late'
    return 'other'

df['loan_status_norm'] = df['loan_status'].apply(normalize_status)

In [18]:
# it will display the first 10 rows, so you can see:

df[['loan_status', 'loan_status_norm']].head(10)

,loan_status,loan_status_norm
0,Current,current
1,Current,current
2,Current,current
3,Current,current
4,Current,current
5,Current,current
6,Current,current
7,Current,current
8,Current,current
9,Current,current


In [21]:
# 4️⃣ Drop rows missing critical fields

df_clean = df[df['grade'].notna()& df['issue_d'].notna()& (df['loan_status_norm'] != 'unknown')].copy()

print("\nAfter cleaning:", df_clean.shape[0], "rows remain")


After cleaning: 2260668 rows remain


In [25]:
# 5️⃣ Flag charged-off loans
df_clean['is_charged_off'] = (df_clean['loan_status_norm'] == 'charged_off').astype(int)



In [42]:
# 6️⃣ Group by loan grade and compute default rate
summary = (
    df_clean
    .groupby('grade')
    .agg(
        loans_count=('loan_amnt', 'count'),           # ✅ fix here
        charged_off_count=('is_charged_off', 'sum'),
        sum_loan_amnt=('loan_amnt', 'sum'),
        avg_loan_amnt=('loan_amnt', 'mean'),
        avg_int_rate=('int_rate', 'mean')
    )
    .reset_index()
)

# Safe calculation of default rate
summary['default_rate'] = summary.apply(
    lambda x: x['charged_off_count'] / x['loans_count'] if x['loans_count'] > 0 else 0,
    axis=1
)


In [44]:
# 7️⃣ Show results
print("\n📊 Default Rate by Loan Grade:")
print(summary.to_string(index=False))


📊 Default Rate by Loan Grade:
grade  loans_count  charged_off_count  sum_loan_amnt  avg_loan_amnt  avg_int_rate  default_rate
    G        12168               4554      248032375   20383.988741     28.074255      0.374260
    F        41800              14356      799410225   19124.646531     25.454203      0.343445
    E       135639              35522     2367318100   17453.078392     21.829848      0.261886
    D       324424              59638     5097344375   15711.983007     18.143304      0.183827
    C       650053              83410     9775551175   15038.083318     14.143793      0.128313
    B       663557              51162     9404817775   14173.338199     10.675819      0.077103
    A       433027              13774     6323641900   14603.343210      7.084558      0.031809


🎯 Why This Metric Matters to the Business

Direct Measure of Credit Risk

Shows the percentage of loans that end in loss to the lender.

A rising default rate signals increased portfolio risk and potential loss exposure.

Portfolio Health Indicator

By tracking default rates by grade (A–G), management can quickly see which loan segments are performing well and which ones are deteriorating.

It turns millions of raw loan records into a simple dashboard metric everyone can understand.